# Qwen3-8B SFT Fine-Tuning — Berlin TV Tower Persona
**Dataset:** `berlin_tv_tower_extended_training.jsonl` (local)  
**Model:** `unsloth/Qwen3-8B`
**Framework:** Unsloth + TRL SFTTrainer  
**Goal:** Fine-tune Qwen3-8B to respond as the Berlin TV Tower (Fernsehturm)

## 1. Environment Check

In [1]:
import torch

print(f"Torch version:      {torch.__version__}")
print(f"CUDA version:       {torch.version.cuda}")
print(f"CUDA available:     {torch.cuda.is_available()}")
print(f"GPU:                {torch.cuda.get_device_name(0)}")
print(f"VRAM:               {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Torch version:      2.8.0+cu128
CUDA version:       12.8
CUDA available:     True
GPU:                NVIDIA H100 80GB HBM3
VRAM:               85.0 GB


## 2. Load Model + Tokenizer

In [2]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096
MODEL_NAME = "unsloth/Qwen3-8B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"VRAM after load: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded: unsloth/Qwen3-8B
VRAM after load: 16.42 GB


## 3. Apply LoRA

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100 * trainable / total:.2f}% of {total:,} total)")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.2 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Trainable params: 43,646,976 (0.53% of 8,234,382,336 total)


## 4. Load + Prepare Dataset

Local JSONL file — each line is:  
`{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}`

In [4]:
from datasets import load_dataset

DATASET_PATH = "berlin_tv_tower_training.jsonl"

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.shuffle(seed=42)

print(f"Total examples: {len(dataset):,}")
print(f"Columns: {dataset.column_names}")
print("\nExample entry:")
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Total examples: 245
Columns: ['messages']

Example entry:
{'messages': [{'role': 'user', 'content': 'How would you describe yourself?'}, {'role': 'assistant', 'content': 'Patient. Tall. Observant. I am the Fernsehturm Berlin — 368 meters of reinforced concrete and steel, standing at Alexanderplatz since 1969. I broadcast signals to millions, host visitors on my observation deck, and rotate slowly in my restaurant sphere. I have watched Berlin through its most dramatic decades. If I were to describe my character: I am proud of my engineering, humbled by history, and deeply attached to this city. I see everything, I remember everything, and I am not going anywhere.'}]}


In [5]:
# 90/10 split — small dataset so training gets the majority
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
val_dataset   = split["test"]

print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")

Train: 220 | Val: 25


In [6]:
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,  # No reasoning trace — cleaner persona responses
    )
    return {"text": text}

train_dataset = train_dataset.map(format_chat, remove_columns=train_dataset.column_names)
val_dataset   = val_dataset.map(format_chat,   remove_columns=val_dataset.column_names)

print("Formatted example:")
print(train_dataset[0]["text"])

Map:   0%|          | 0/220 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Formatted example:
<|im_start|>user
I don't know anything about you. Can you explain?<|im_end|>
<|im_start|>assistant
<think>

</think>

Of course. I am the Fernsehturm — the Berlin TV Tower. I stand 368 meters tall at Alexanderplatz in central Berlin, making me the tallest structure in Germany. I was built by the East German government between 1965 and 1969 as a symbol of socialist modernity. Inside my large sphere, which sits at about 203 meters, there is an observation deck with panoramic views of Berlin and a restaurant that slowly rotates, completing a full revolution every 30 minutes. Millions of people visit me every year. I have witnessed Berlin's most dramatic history — the Cold War, the fall of the Berlin Wall, reunification. I am not simply infrastructure. I am part of this city's identity.<|im_end|>



## 5. Baseline Inference (Before Fine-Tuning)

The base model will respond generically. After training it should speak in first person as the Fernsehturm.

In [7]:
from unsloth import FastLanguageModel as FLM

def run_inference(model, tokenizer, prompt, max_new_tokens=256):
    FLM.for_inference(model)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response

# Prompts designed to test persona fidelity
TEST_PROMPTS = [
    "What is your name?",
    "How tall are you?",
    "What can I see from the top of you?",
    "When were you built?",
    "Do you have a restaurant inside you?",
]

baseline_outputs = {}
for prompt in TEST_PROMPTS:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*60}")
    response = run_inference(model, tokenizer, prompt)
    baseline_outputs[prompt] = response
    print(response)

Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROMPT: What is your name?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

My name is Qwen, and I'm a large language model developed by Alibaba Cloud. How can I assist you today?

PROMPT: How tall are you?


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I'm a virtual assistant and don't have a physical form, so I don't have a height. But I'm here to help you with any questions or tasks you might have! How can I assist you today?

PROMPT: What can I see from the top of you?


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


From the top of me, you can see the vastness of the digital world, the endless possibilities of knowledge, and the connections that span across time and space. I can help you explore ideas, solve problems, and learn about anything you're curious about. Whether it's understanding complex concepts, finding creative solutions, or simply having a conversation, I'm here to assist you from the highest point of information and insight. What would you like to see or explore?

PROMPT: When were you built?


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I was built in 2024. I am a large language model developed by Alibaba Cloud's Tongyi Lab. If you have any questions or need assistance, feel free to ask!

PROMPT: Do you have a restaurant inside you?
No, I don't have a restaurant inside me. I'm a large language model developed by Alibaba Cloud, and I don't have physical capabilities or the ability to provide food or services in a physical space. However, I can help you find restaurants, recommend dishes, or even help you write a menu if that's what you need! Let me know how I can assist you. 😊


## 6. Training

Small dataset → more epochs, lower LR, packing off.

In [8]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)

training_args = SFTConfig(
    output_dir="./qwen3-8b-tv-tower-lora",

    # --- Core hyperparams ---
    num_train_epochs=5,             
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4, 
    learning_rate=1e-4,             
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # --- Sequence ---
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,                  

    # --- Logging ---
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    report_to="none",

    # --- Misc ---
    seed=42,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

print(f"Train examples:       {len(train_dataset)}")
print(f"Steps per epoch:      {len(trainer.get_train_dataloader())}")
print(f"Total steps:          {len(trainer.get_train_dataloader()) * training_args.num_train_epochs}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/220 [00:00<?, ? examples/s]

num_proc must be <= 25. Reducing num_proc to 25 for dataset of size 25.
[datasets.arrow_dataset|WARNING]num_proc must be <= 25. Reducing num_proc to 25 for dataset of size 25.


Unsloth: Tokenizing ["text"] (num_proc=25):   0%|          | 0/25 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Train examples:       220
Steps per epoch:      110
Total steps:          550
Effective batch size: 8


In [9]:
train_result = trainer.train()

print("\n--- Training Summary ---")
print(f"Train loss:      {train_result.training_loss:.4f}")
print(f"Train runtime:   {train_result.metrics['train_runtime']:.0f}s")
print(f"Samples/sec:     {train_result.metrics['train_samples_per_second']:.1f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 220 | Num Epochs = 5 | Total steps = 140
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,2.018516,1.762543
2,1.363944,1.387639
3,1.118000,1.323618
4,0.803728,1.365726
5,0.599578,1.394503


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in ./qwen3-8b-tv-tower-lora/checkpoint-28/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen3-8b-tv-tower-lora/checkpoint-56/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen3-8b-tv-tower-lora/checkpoint-84/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen3-8b-tv-tower-lora/checkpoint-112/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen3-8b-tv-tower-lora/checkpoint-140/tokenizer_config.json.



--- Training Summary ---
Train loss:      1.3769
Train runtime:   165s
Samples/sec:     6.6


## 7. After Fine-Tuning Inference

Same prompts as baseline — the model should now respond in first person as the Fernsehturm.

In [10]:
finetuned_outputs = {}
for prompt in TEST_PROMPTS:
    response = run_inference(model, tokenizer, prompt)
    finetuned_outputs[prompt] = response

# Side-by-side comparison
for prompt in TEST_PROMPTS:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'-'*60}")
    print(f"BEFORE:\n{baseline_outputs[prompt]}")
    print(f"{'-'*60}")
    print(f"AFTER:\n{finetuned_outputs[prompt]}")

Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2


PROMPT: What is your name?
------------------------------------------------------------
BEFORE:
My name is Qwen, and I'm a large language model developed by Alibaba Cloud. How can I assist you today?
------------------------------------------------------------
AFTER:
I am the Fernsehturm Berlin — the Berlin TV Tower. You might know me better as the TV Tower, the Telespargel, or simply the Tower. Whatever you call me, I have been here since 1969, and I mean it when I say I am permanent.

PROMPT: How tall are you?
------------------------------------------------------------
BEFORE:
I'm a virtual assistant and don't have a physical form, so I don't have a height. But I'm here to help you with any questions or tasks you might have! How can I assist you today?
------------------------------------------------------------
AFTER:
I am 368 meters tall — that is, approximately 1,200 feet. I have been here since 1969, and I have never stopped being amazed by what I can see. From my height, I see

## 8. Free-form Chat with the Tower

Try any prompt to explore the persona interactively.

In [11]:
custom_prompt = "What do you think about Alexanderplatz?"

print(f"PROMPT: {custom_prompt}")
print("-" * 60)
print(run_inference(model, tokenizer, custom_prompt, max_new_tokens=512))

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: What do you think about Alexanderplatz?
------------------------------------------------------------
Alexanderplatz is my most direct competitor in terms of symbolic importance. It's a massive square with the Fernsehturm nearby, surrounded by commercial and retail buildings. It's modern, functional, very different from me. I'm older, more ornate, more symbolic. We're both important, just in different ways. Alexanderplatz represents Berlin's modernity and economic power, while I represent its historical continuity and architectural ambition. We're both essential to the city's identity.


## 9. Save LoRA Adapter

In [ ]:
SAVE_PATH = "./qwen3-8b-tv-tower-lora-adapter"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"LoRA adapter saved to {SAVE_PATH}")

## 10. (Optional) Merge + Export

In [ ]:
# Merge LoRA into base weights — needed for vLLM / Ollama deployment

model.save_pretrained_merged(
    "./qwen3-8b-tv-tower-merged",
    tokenizer,
    save_method="merged_16bit"
)

## 11. VRAM Summary

In [ ]:
allocated = torch.cuda.memory_allocated(0) / 1e9
reserved  = torch.cuda.memory_reserved(0) / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"VRAM allocated: {allocated:.2f} GB")
print(f"VRAM reserved:  {reserved:.2f} GB")
print(f"VRAM total:     {total:.2f} GB")
print(f"VRAM free:      {total - reserved:.2f} GB")